# FlashEats — Session 7 challenge: modelling the business workflow with data

**Client question:** *"Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI."*

Project KPI: **reduce late delivery rate**. Source discovery, retrieval and cleaning are not repeated here — the modelled tables produced by `run_pipeline.py` are used (`data/processed/`).

In [1]:
# --- setup: locate the repository root and load the pipeline package -------------
import sys, json, sqlite3, subprocess, time
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "source_systems").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
BASE = ROOT / "source_systems"            # the client systems snapshot (read-only)
NB_OUT = ROOT / "notebooks" / "output"      # scratch outputs of this notebook
NB_OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90)
print("ROOT:", ROOT, "| pack exists:", (BASE / "database" / "flasheats.db").exists())

ROOT: B:\Trimester-9\FDE\assignment-2 | pack exists: True


In [2]:
P = ROOT / "data" / "processed"
if not (P / "fact_order.csv").exists():
    raise FileNotFoundError("Run `python run_pipeline.py --start-api` first - this notebook reads the modelled tables from data/processed/")
dt_cols = ["created_at", "promised_eta", "pickup_at", "actual_delivery_at", "assigned_at", "reassigned_at", "estimated_pickup_at", "current_delivery_eta", "ev_delivered_at"]
fact_order = pd.read_csv(P / "fact_order.csv", parse_dates=[c for c in dt_cols])
fact_event = pd.read_csv(P / "fact_event.csv", parse_dates=["event_time"])
fact_interaction = pd.read_csv(P / "fact_interaction.csv", parse_dates=["interaction_at"])
fact_intervention = pd.read_csv(P / "fact_intervention.csv", parse_dates=["intervention_at"])
order_journey = pd.read_csv(P / "order_journey.csv", parse_dates=["created_at", "promised_eta", "actual_delivery_at", "first_intervention_at", "first_ticket_at", "first_app_action_at"])
dim_restaurant = pd.read_csv(P / "dim_restaurant.csv")
for name, df in [("fact_order", fact_order), ("fact_event", fact_event), ("fact_interaction", fact_interaction), ("fact_intervention", fact_intervention), ("order_journey", order_journey)]:
    print(f"{name:18s} {df.shape}")

fact_order         (1600, 52)
fact_event         (21518, 7)
fact_interaction   (2566, 8)
fact_intervention  (430, 9)
order_journey      (1600, 33)


# Challenge 1 — Reconstruct the order lifecycle

Three orders — one on time, one late, one with an intervention — as `event_time | event_type | actor | source_system`, merged from every source and sorted by time.

In [3]:
def build_order_timeline(order_id):
    t = fact_event[fact_event["order_id"] == order_id].sort_values("event_time")
    return t[["event_time", "event_type", "actor_type", "actor_id", "source_system", "detail"]].reset_index(drop=True)

on_time_order = fact_order[fact_order["outcome_bucket"] == "delivered_on_time"]["order_id"].iloc[0]
late_order = fact_order[(fact_order["outcome_bucket"] == "delivered_late") & (fact_order["delay_min"] > 20)]["order_id"].iloc[0]
intervention_order = order_journey[(order_journey["intervention_count"] > 0) & (order_journey["ticket_count"] > 0) & (order_journey["late_flag"] == 1)]["order_id"].iloc[0]
for label, oid in [("ON TIME", on_time_order), ("LATE", late_order), ("INTERVENTION + TICKET + STILL LATE", intervention_order)]:
    row = fact_order.set_index("order_id").loc[oid]
    print(f"=== {label}: {oid} | delay {row['delay_min']} min | promised {row['promised_eta']} | delivered {row['actual_delivery_at']}")
    display(build_order_timeline(oid))

=== ON TIME: O00003 | delay -4.41 min | promised 2026-08-01 20:17:52.941967 | delivered 2026-08-01 20:13:28.061191


,event_time,event_type,actor_type,actor_id,source_system,detail
0,2026-08-01 19:35:00.000000,ORDER_CREATED,customer,C0855,order_platform,NaN
1,2026-08-01 19:35:00.002443,DRIVER_APP_ASSIGNED,driver,D039,driver_app,NaN
2,2026-08-01 19:35:00.002443,DRIVER_ASSIGNED,dispatch,D039,dispatch_api,NaN
3,2026-08-01 19:44:37.017130,DRIVER_APP_GPS_PING,driver,D039,driver_app,NaN
4,2026-08-01 19:48:00.000000,RESTAURANT_PREPARING,restaurant,R010,restaurant_status_feed,preparing
5,2026-08-01 19:53:00.000000,PICKUP_ESTIMATED,dispatch,D039,dispatch_api,NaN
6,2026-08-01 19:54:14.031817,DRIVER_APP_GPS_PING,driver,D039,driver_app,NaN
7,2026-08-01 19:57:50.820366,DRIVER_APP_PICKED_UP,driver,D039,driver_app,NaN
8,2026-08-01 19:57:50.820366,PICKED_UP,driver,D039,order_platform,NaN
9,2026-08-01 20:03:51.046504,DRIVER_APP_GPS_PING,driver,D039,driver_app,NaN


=== LATE: O00005 | delay 21.02 min | promised 2026-08-06 21:54:48 | delivered 2026-08-06 22:15:49.290713


,event_time,event_type,actor_type,actor_id,source_system,detail
0,2026-08-06 20:35:00.000000,ORDER_CREATED,customer,C0040,order_platform,NaN
1,2026-08-06 20:36:55.346822,DRIVER_APP_ASSIGNED,driver,D047,driver_app,NaN
2,2026-08-06 20:36:55.346822,DRIVER_ASSIGNED,dispatch,D047,dispatch_api,NaN
3,2026-08-06 20:53:00.000000,PICKUP_ESTIMATED,dispatch,D047,dispatch_api,NaN
4,2026-08-06 21:09:53.328119,DRIVER_APP_GPS_PING,driver,D047,driver_app,NaN
5,2026-08-06 21:18:38.024280,DRIVER_APP_PICKED_UP,driver,D047,driver_app,NaN
6,2026-08-06 21:18:38.024280,PICKED_UP,driver,D047,order_platform,NaN
7,2026-08-06 21:42:51.309416,DRIVER_APP_GPS_PING,driver,D047,driver_app,NaN
8,2026-08-06 22:15:49.290713,DELIVERED,driver,D047,order_platform,NaN
9,2026-08-06 22:15:49.290713,DRIVER_APP_DELIVERED,driver,D047,driver_app,NaN


=== INTERVENTION + TICKET + STILL LATE: O00058 | delay 2.7 min | promised 2026-08-05 21:06:02.914275 | delivered 2026-08-05 21:08:44.647933


,event_time,event_type,actor_type,actor_id,source_system,detail
0,2026-08-05 20:01:00.000000,ORDER_CREATED,customer,C0604,order_platform,NaN
1,2026-08-05 20:01:56.036115,DRIVER_APP_ASSIGNED,driver,D007,driver_app,NaN
2,2026-08-05 20:01:56.036115,DRIVER_ASSIGNED,dispatch,D007,dispatch_api,NaN
3,2026-08-05 20:05:00.000000,APP_ETA_VIEWED,customer,C0604,customer_app,mobile_app
4,2026-08-05 20:12:00.000000,RESTAURANT_HANDED_OFF,restaurant,R030,restaurant_status_feed,handed_off
5,2026-08-05 20:18:38.189069,DRIVER_APP_GPS_PING,driver,D007,driver_app,NaN
6,2026-08-05 20:19:00.000000,PICKUP_ESTIMATED,dispatch,D007,dispatch_api,NaN
7,2026-08-05 20:32:28.460434,DRIVER_APP_PICKED_UP,driver,D007,driver_app,NaN
8,2026-08-05 20:32:28.460434,PICKED_UP,driver,D007,order_platform,NaN
9,2026-08-05 20:35:20.342024,DRIVER_APP_GPS_PING,driver,D007,driver_app,NaN


The timeline makes the gaps visible: between `DRIVER_ASSIGNED` and `PICKED_UP` there is no *arrived at restaurant* event, so the wait at the restaurant and the ride to it are one block; cancelled orders have no terminal event at all.

# Challenge 2 — Define the canonical project model

| Table | Grain (one row per…) | Primary key | Important foreign keys | Supports |
|---|---|---|---|---|
| `dim_customer` | customer | `customer_id` | — | customer → orders |
| `dim_restaurant` | restaurant | `restaurant_id` | — | restaurant attribution (with `coords_valid`) |
| `dim_driver` | driver | `driver_id` | — | driver attribution |
| `fact_order` | order | `order_id` | `customer_id`, `restaurant_id`, `driver_id` (final, from Dispatch), `original_driver_id` | order → outcome (`late_flag`, `delay_min`, `outcome_bucket`), stage durations, dq flags, KPI population |
| `fact_event` | lifecycle event | (`order_id`, `event_time`, `event_type`, `source_system`) | `order_id` | order → events/states |
| `fact_interaction` | customer interaction (app action or ticket) | `interaction_id` | `order_id`, `customer_id` | order → customer interactions, order → support interactions |
| `fact_intervention` | intervention | `intervention_id` | `order_id` | order → interventions |
| `order_journey` | order | `order_id` | as `fact_order` | interaction → intervention → outcome in one row |

Relationships: customer 1—n order; order 1—n event; order 1—n interaction; order 1—n intervention; order 1—1 outcome; restaurant 1—n order; driver 1—n order (with the reassignment kept as an event and as `original_driver_id`).

**Why this beats mirroring every source table:** the sources are organised by *system* (orders DB, dispatch, driver app, support desk, app analytics) and disagree on grain, spelling and even on who the driver is. The model is organised around the **workflow the KPI lives in**: every one-to-many table is aggregated to order grain *before* it touches the order, so the denominator cannot inflate, and every business question ("late orders with support contact and no intervention") is one filter on `order_journey` instead of a five-way join that nobody can audit.

In [4]:
for name, df, key in [("fact_order", fact_order, "order_id"), ("fact_interaction", fact_interaction, "interaction_id"), ("fact_intervention", fact_intervention, "intervention_id"), ("order_journey", order_journey, "order_id")]:
    print(f"{name:18s} rows={len(df):6d} unique {key}={df[key].nunique():6d} -> grain OK: {df[key].is_unique}")
print("every order has exactly one journey row:", set(order_journey["order_id"]) == set(fact_order["order_id"]))
print(json.dumps(json.loads((P / "join_checks.json").read_text()), indent=1))

fact_order         rows=  1600 unique order_id=  1600 -> grain OK: True
fact_interaction   rows=  2566 unique interaction_id=  2566 -> grain OK: True
fact_intervention  rows=   430 unique intervention_id=   430 -> grain OK: True
order_journey      rows=  1600 unique order_id=  1600 -> grain OK: True
every order has exactly one journey row: True
[
 {
  "merge": "fact_order <- dispatch",
  "left_rows": 1600,
  "right_rows": 1600,
  "result_rows": 1600,
  "right_duplicates_dropped": 0,
  "matched_rows": 1600,
  "row_count_preserved": true
 },
 {
  "merge": "fact_order <- driver_events (aggregated)",
  "left_rows": 1600,
  "right_rows": 1600,
  "result_rows": 1600,
  "right_duplicates_dropped": 0,
  "matched_rows": 1600,
  "row_count_preserved": true
 },
 {
  "merge": "order_journey <- app_actions (aggregated)",
  "left_rows": 1600,
  "right_rows": 1100,
  "result_rows": 1600,
  "right_duplicates_dropped": 0,
  "matched_rows": 1100,
  "row_count_preserved": true
 },
 {
  "merge": "order_jo

# Challenge 3 — Build interaction → intervention → outcome

One order-level table: `order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min` (built by aggregating the one-to-many tables first).

In [5]:
cols = ["order_id", "customer_id", "support_opened", "cancel_attempted", "ticket_count", "intervention_count", "intervention_types", "final_status", "late_flag", "delay_min", "outcome_bucket"]
display(order_journey[cols].head(8))
pop = order_journey[order_journey["kpi_population"]]
late = pop[pop["late_flag"] == 1]
print("1. Late orders with a support interaction (ticket or SUPPORT_OPENED):", int(late["support_contact"].sum()), "of", len(late), f"({100*late['support_contact'].mean():.1f}%)")
print("2. Orders that received an intervention:", int((order_journey["intervention_count"] > 0).sum()), "of", len(order_journey))
print("3. Most common intervention:", fact_intervention["intervention_type"].value_counts().head(1).to_dict())
frustrated_no_iv = order_journey[order_journey["frustrated_no_intervention"]]
print("4. Frustrated journeys (support contact) with NO intervention:", len(frustrated_no_iv), "| of which late:", int((frustrated_no_iv["late_flag"] == 1).sum()))
display(frustrated_no_iv[["order_id", "ticket_categories", "support_opened", "late_flag", "delay_min"]].sort_values("delay_min", ascending=False).head(8))

,order_id,customer_id,support_opened,cancel_attempted,ticket_count,intervention_count,intervention_types,final_status,late_flag,delay_min,outcome_bucket
0,O00001,C0168,True,False,0,1,DRIVER_REASSIGNMENT,delivered,1.0,10.82,delivered_late
1,O00002,C0043,False,False,0,0,NaN,delivered,1.0,3.97,delivered_late
2,O00003,C0855,False,False,0,0,NaN,delivered,0.0,-4.41,delivered_on_time
3,O00004,C0229,False,False,0,0,NaN,cancelled,NaN,NaN,cancelled
4,O00005,C0040,False,False,0,0,NaN,delivered,1.0,21.02,delivered_late
5,O00006,C0152,False,False,0,0,NaN,delivered,1.0,7.30,delivered_late
6,O00007,C0875,False,False,0,0,NaN,delivered,1.0,4.27,delivered_late
7,O00008,C0223,True,False,0,1,CUSTOMER_CREDIT,delivered,1.0,26.86,delivered_late


1. Late orders with a support interaction (ticket or SUPPORT_OPENED): 257 of 837 (30.7%)
2. Orders that received an intervention: 430 of 1600
3. Most common intervention: {'DRIVER_REASSIGNMENT': 155}
4. Frustrated journeys (support contact) with NO intervention: 218 | of which late: 191


,order_id,ticket_categories,support_opened,late_flag,delay_min
193,O00194,ready_but_waiting,False,1.0,38.34
736,O00737,NaN,True,1.0,36.28
302,O00303,status_mismatch,False,1.0,33.16
210,O00211,restaurant_delay,True,1.0,32.38
1234,O01235,eta_changed,False,1.0,32.00
976,O00977,status_mismatch,True,1.0,31.92
994,O00995,driver_not_moving,True,1.0,31.72
1196,O01197,driver_not_moving,False,1.0,31.61


# Challenge 4 — Select 3–5 business metrics

| # | Metric | Formula | Grain | Why it matters | Link to KPI |
|---|---|---|---|---|---|
| M1 | **Late delivery rate** (outcome) | late orders / validated delivered orders | order | the project KPI itself | direct |
| M2 | **Lateness severity** (outcome) | median delay among late orders; P90 delay over all deliveries | order | tells whether the problem is "many small misses" or "few disasters" | severity of the KPI |
| M3 | **Share of lateness accumulated before pickup** (workflow) | Σ max(pickup − estimated_pickup, 0) / Σ all positive stage overruns, late orders | order → stage | says *where* to intervene (kitchen/dispatch vs road) | the driver of the KPI |
| M4 | **Support contact rate on late orders** (interaction) | late orders with ticket or SUPPORT_OPENED / late orders | order | customer impact and early-warning signal | consequence of the KPI |
| M5 | **Intervention coverage & late rate with vs without intervention** (intervention) | orders with ≥1 intervention / orders; late rate per group and per type | order | does the ops team reach the orders that need help, and does it coincide with better outcomes | lever on the KPI |

All five (plus supporting variants) are computed by `metrics.py` and written to `output/metrics.csv`.

In [6]:
metrics = pd.read_csv(ROOT / "output" / "metrics.csv")
for c in ["numerator", "denominator"]:      # show counts as integers, keep minute sums as they are
    metrics[c] = pd.to_numeric(metrics[c], errors="coerce").map(lambda v: int(v) if pd.notna(v) and float(v).is_integer() else v)
display(metrics[metrics["metric_id"].isin(["M1", "M1a", "M1b", "M2", "M2a", "M3", "M3a", "M3b", "M4", "M4a", "M5", "M5a", "M5b"])][["metric_id", "metric", "category", "value", "unit", "numerator", "denominator"]])

,metric_id,metric,category,value,unit,numerator,denominator
0,M1,Late delivery rate (validated population),outcome,56.33,%,837.0,1486.0
1,M1a,Late delivery rate (historical dashboard definition),outcome,56.39,%,843.0,1495.0
2,M1b,Meaningfully late rate (> 10 min),outcome,23.15,%,344.0,1486.0
4,M2,Median lateness among late orders,outcome,8.0,min,NaN,837.0
5,M2a,P90 delay across all validated deliveries,outcome,17.9,min,NaN,1486.0
7,M3,Share of lateness accumulated before pickup (vs in transit),workflow,98.91,%,13150.8,13296.1
8,M3a,Median pre-pickup overrun on late orders,workflow,13.9,min,NaN,837.0
9,M3b,Median transit overrun on late orders,workflow,-5.9,min,NaN,837.0
10,M4,Support contact rate on late orders,interaction,30.7,%,257.0,837.0
11,M4a,Support contact rate on on-time orders,interaction,5.24,%,34.0,649.0


# Challenge 5 — Investigate the workflow with joins and aggregations

In [7]:
def late_rate(df):
    return round(100 * (df["late_flag"] == 1).mean(), 1) if len(df) else None

# A. Do orders with support interactions have higher delay?
a = pop.groupby("support_contact")["delay_min"].agg(orders="size", median_delay="median", mean_delay="mean").round(1)
a["late_rate_%"] = pop.groupby("support_contact").apply(late_rate, include_groups=False)
print("A. delay by support contact"); display(a)

# B. Late rate with vs without intervention
b = pop.groupby("has_intervention")["late_flag"].agg(orders="size", late=lambda s: int((s == 1).sum()))
b["late_rate_%"] = (100 * b["late"] / b["orders"]).round(1)
print("B. late rate with vs without intervention"); display(b)

# C. Which intervention type is associated with the lowest late rate?
c = fact_intervention.merge(pop[["order_id", "late_flag", "delay_min"]], on="order_id", how="inner")
c = c.groupby("intervention_type").agg(orders=("order_id", "nunique"), late_rate_pct=("late_flag", lambda s: round(100 * (s == 1).mean(), 1)), median_delay=("delay_min", "median"), before_promise_pct=("before_promised_eta", lambda s: round(100 * s.mean(), 1))).sort_values("late_rate_pct")
print("C. late rate by intervention type"); display(c)

# D. Which restaurants contribute the most late orders?
d = fact_order[fact_order["kpi_population"]].merge(dim_restaurant[["restaurant_id", "cuisine", "manual_status_updates"]], on="restaurant_id", how="left")
d = d.groupby(["restaurant_id", "cuisine"]).agg(orders=("order_id", "size"), late=("late_flag", "sum"), late_rate_pct=("late_flag", lambda s: round(100 * s.mean(), 1)), median_delay=("delay_min", "median")).reset_index()
d["share_of_all_late_%"] = (100 * d["late"] / d["late"].sum()).round(1)
print("D. top restaurants by late orders (share of all late orders)"); display(d.sort_values("late", ascending=False).head(10))

# E. Journeys with support interaction + intervention + still late
e = order_journey[order_journey["frustrated_intervened_still_late"]]
print("E. support contact + intervention + still late:", len(e)); display(e[["order_id", "ticket_categories", "intervention_types", "delay_min"]].sort_values("delay_min", ascending=False).head(8))

A. delay by support contact


,orders,median_delay,mean_delay,late_rate_%
support_contact,,,,
False,1195,-0.3,0.6,48.5
True,291,12.2,12.5,88.3


B. late rate with vs without intervention


,orders,late,late_rate_%
has_intervention,,,
False,1087,612,56.3
True,399,225,56.4


C. late rate by intervention type


,orders,late_rate_pct,median_delay,before_promise_pct
intervention_type,,,,
PRIORITY_DISPATCH,86,51.2,0.33,100.0
RESTAURANT_CONTACT,105,52.4,0.58,100.0
DRIVER_REASSIGNMENT,147,57.8,1.60,100.0
CUSTOMER_CREDIT,61,67.2,2.93,0.0


D. top restaurants by late orders (share of all late orders)


,restaurant_id,cuisine,orders,late,late_rate_pct,median_delay,share_of_all_late_%
49,R050,Cafe,35,22.0,62.9,2.210,2.6
23,R024,South Indian,25,20.0,80.0,6.190,2.4
50,R051,Cafe,29,20.0,69.0,4.710,2.4
22,R023,Pizza,32,20.0,62.5,2.835,2.4
44,R045,Cafe,29,19.0,65.5,6.660,2.3
29,R030,South Indian,37,19.0,51.4,0.250,2.3
3,R004,South Indian,29,19.0,65.5,6.820,2.3
10,R011,Chinese,31,19.0,61.3,3.450,2.3
24,R025,South Indian,28,18.0,64.3,2.965,2.2
17,R018,South Indian,27,18.0,66.7,4.020,2.2


E. support contact + intervention + still late: 66


,order_id,ticket_categories,intervention_types,delay_min
197,O00198,eta_changed,RESTAURANT_CONTACT,35.23
496,O00497,restaurant_delay,PRIORITY_DISPATCH,31.78
612,O00613,status_mismatch,PRIORITY_DISPATCH,27.18
7,O00008,NaN,CUSTOMER_CREDIT,26.86
301,O00302,restaurant_delay,PRIORITY_DISPATCH,26.32
543,O00544,late_delivery,DRIVER_REASSIGNMENT,24.92
861,O00862,late_delivery,DRIVER_REASSIGNMENT,24.10
538,O00539,restaurant_delay,DRIVER_REASSIGNMENT,23.28


**What each answer tells the business — and what it does NOT prove**

- **A.** Orders with a support contact have a median delay of ~12 min vs ~−0.3 min without, and a late rate of ~88 % vs ~49 %. *Tells:* complaints are a reliable lagging signal of real lateness. *Does not prove:* that contacting support changes the outcome — the customer contacts support *because* the order is late.
- **B.** Late rate is ~56 % with and ~56 % without an intervention. *Tells:* interventions are not associated with a lower late rate overall. *Does not prove:* that interventions are useless — they are targeted at late-risk orders (selection effect), so "no difference" may mean they pulled a 70 % risk down to 56 %.
- **C.** `PRIORITY_DISPATCH` and `RESTAURANT_CONTACT` show the lowest late rates (~51–52 %), `CUSTOMER_CREDIT` the highest (67 %). *Tells:* credits are issued *after* a bad outcome (0 % before the promised ETA), the other three act before it. *Does not prove:* which intervention causes better outcomes — no randomisation, different triggers per type.
- **D.** No single restaurant dominates: the top contributor has ~2.6 % of all late orders; a handful (e.g. R024, 80 % late) stand out on rate. *Tells:* lateness is systemic, not a "few bad kitchens" story. *Does not prove:* kitchen fault — restaurant prep time is not observed (no arrival event).
- **E.** 66 journeys had a support contact **and** an intervention **and** were still late. *Tells:* the recovery loop exists but is late or weak. *Does not prove:* that earlier intervention would have saved them.

# Challenge 6 — Connect the model to the KPI

```
PROJECT KPI          reduce late delivery rate (M1)
   ↑
OUTCOME METRICS      M1 late rate · M2 severity (median / P90 delay)
   ↑
WORKFLOW METRICS     M3 pre-pickup overrun share · stage durations (dispatch wait, assign→pickup, transit) · ETA revision
   ↑                 M4 support contact rate (customer reaction)
INTERVENTIONS        M5 coverage · late rate by type (reassignment, restaurant contact, priority dispatch, credit)
   ↑
DATA SOURCES/EVENTS  orders DB (created, promised, pickup, delivered) · Dispatch API (assigned, estimated pickup, reassigned)
                     driver app (assigned, GPS, picked_up, delivered) · restaurant feed (preparing/ready/handed_off)
                     customer app (ETA_VIEWED, SUPPORT_OPENED, CANCEL_ATTEMPTED) · support desk (tickets) · interventions log
```

1. **Directly controllable by operations:** dispatch wait (assignment speed), intervention coverage and timing (M5), which orders get priority dispatch / restaurant contact, and the ETA promise itself (planned pickup and transit windows).
2. **Outcomes:** M1, M2, M4 — they move only when the controllable levers move.
3. **Missing event that limits the model most:** *driver arrived at restaurant*. Without it, M3's "pre-pickup overrun" cannot be split into kitchen time and rider time, so the biggest bucket of lateness (≈99 % of overrun minutes) cannot be assigned to an owner.
4. **What to instrument next:** the arrival event (geofence or a tap in the driver app), a mandatory delivery timestamp on the `delivered` transition, reassignment propagation into the orders table, and a cancellation timestamp.

**Final deliverable — the smallest useful model:** 3 dimensions, 4 facts, 1 order-grain journey table; key events created → assigned → (arrived?) → picked up → delivered plus interactions and interventions; 5 metrics linked to the KPI; one modelling limitation — the pre-pickup stage is a black box.

In [8]:
print("Session 7 notebook complete.")

Session 7 notebook complete.
